# Структура проекта и окружение Python

Когда программа растёт, её удобнее разделить на несколько файлов. А чтобы разные проекты не мешали пакетам друг друга, для каждого проекта можно создать своё виртуальное окружение.

В этом уроке разберём структуру папок, импорты, пакеты, `venv` и `pip`, а затем соберём маленький проект из нескольких модулей.

> Примеры с `%%writefile` используют команду Jupyter и записывают учебные файлы в текущую папку блокнота. Команды установки пакетов выполняй в терминале только при необходимости.

## Карта урока

1. Зачем делить программу на файлы
2. Импортируем функции из соседнего модуля
3. Защищаем точку входа через `if __name__ == "__main__"`
4. Пакет и запуск через `python -m`
5. Виртуальное окружение `venv`
6. Устанавливаем и сохраняем зависимости через `pip`
7. Практика: небольшой проект


## 1. Структура проекта

Небольшой проект — это папка с кодом и файлами, которые относятся к одной программе. Например, функции можно хранить отдельно от главной части программы.

![Структура проекта и связь между импортирующими файлами](assets/project_import.png)

Старайся не называть свои файлы именами стандартных модулей. Например, файл `json.py` рядом с программой может помешать команде `import json` найти стандартную библиотеку.


## 2. Импорт из соседнего файла

Создадим файл `lesson_tools.py` с функцией. `%%writefile` — это команда Jupyter, которая сохраняет текст ячейки в файл.


In [ ]:
%%writefile lesson_tools.py
def greet(name):
    return f"Привет, {name}!"


def make_line(symbol, count):
    return symbol * count


Теперь Python может импортировать функцию по имени файла (без `.py`). Оба файла должны находиться в доступной для импорта папке; в этом упражнении они лежат рядом с блокнотом.


In [ ]:
from lesson_tools import greet, make_line

print(greet("Аня"))
print(make_line("-", 12))

Можно импортировать модуль целиком и вызывать функции через его имя:


In [ ]:
import lesson_tools

print(lesson_tools.greet("Борис"))

Если используешь `from module import name`, в коде будут доступны только перечисленные имена. Если импортируешь модуль через `import module`, имя модуля помогает понять, откуда взялась функция.

### Импорт и повторный запуск ячеек

В интерактивном блокноте Python сохраняет уже импортированные модули в памяти. Если ты изменил `lesson_tools.py`, перезапусти ядро и снова запусти ячейки или используй `importlib.reload()`.


### Импорт выполняет команды модуля

При первом обычном импорте Python выполняет верхний уровень модуля: определения функций создают функции, а обычные `print()` и `input()` запускаются сразу. Поэтому ввод и запуск программы помещают в `main()` под проверкой `if __name__ == "__main__":`.

После изменения модуля повторный `import` обычно берёт его из кеша. Для учебного эксперимента перезапусти ядро блокнота; простого повторного запуска ячейки импорта может быть недостаточно.

**Проверка.** Добавь временный `print("Модуль загружен")` в свой модуль и импортируй его дважды. После перезапуска ядра сообщение появится снова. Затем убери отладочную печать.


In [ ]:
import importlib
import lesson_tools

importlib.reload(lesson_tools)
print(lesson_tools.greet("Вера"))

## 3. Точка входа программы

Файл можно запустить напрямую или импортировать из другого файла. Проверка `if __name__ == "__main__":` запускает нижний блок только при прямом запуске файла.

Создадим файл, который можно запускать и импортировать.


In [ ]:
%%writefile lesson_main.py
from lesson_tools import greet


def main():
    print(greet("Мир Python"))


if __name__ == "__main__":
    main()


При прямом запуске `lesson_main.py` значение `__name__` равно `"__main__"`, и функция `main()` вызывается. При импорте из другого файла условие ложно — программа не запускается сама.

Из терминала команда выглядит так:


```text
python lesson_main.py
```

На некоторых компьютерах команду Python 3 запускают как `python3 lesson_main.py`. Выбери команду, которая соответствует установленному Python.


Проверим импорт `lesson_main`: функция будет доступна, но приветствие само не напечатается.


In [ ]:
import lesson_main

print("Импорт завершён; main() автоматически не запускалась.")

## 4. Пакеты и запуск через `python -m`

Пакет помогает сгруппировать связанные модули. Для учебного проекта можно создать такую структуру:

```text
quiz_project/
├── quiz/
│   ├── __init__.py
│   ├── main.py
│   └── questions.py
└── README.md
```

Файл `__init__.py` показывает, что папка `quiz` задумана как пакет. В простом учебном проекте можно оставить его пустым.


Чтобы запустить модуль `main.py` внутри пакета из папки `quiz_project`, используй команду:

```text
python -m quiz.main
```

Запускай её из корня проекта — папки, внутри которой лежит `quiz`. Команда `-m` запускает модуль как часть пакета, поэтому импорты внутри него работают предсказуемее.


Внутри `quiz/main.py` можно импортировать функцию из соседнего модуля пакета так:

```python
from .questions import get_question
```

Точка в начале — относительный импорт: модуль ищется в том же пакете. Если запускаешь пакетный модуль, используй `python -m quiz.main`, а не запускай файл `quiz/main.py` напрямую.


### Мини-практика: разделим викторину

1. В `questions.py` создай функцию, которая возвращает вопрос и ответ.
2. В `main.py` импортируй её.
3. Вызови `main()` только в блоке `if __name__ == "__main__":`.

Пока можно потренироваться на двух файлах рядом, как в примере выше. Структуру пакета попробуй создать в отдельной папке проекта.


## 5. Виртуальное окружение `venv`

Виртуальное окружение — отдельная папка с интерпретатором и установленными пакетами для одного проекта. Это позволяет проектам использовать разные версии сторонних библиотек.

Открой терминал в папке проекта и создай окружение:

```text
python -m venv .venv
```

Если команда `python` не запускает Python 3, попробуй `python3 -m venv .venv`.


### Активируем окружение

В macOS или Linux:

```text
source .venv/bin/activate
```

В Windows PowerShell:

```text
.venv\Scripts\Activate.ps1
```

В Windows Command Prompt:

```text
.venv\Scripts\activate.bat
```

После активации в начале строки терминала обычно появляется имя окружения, например `(.venv)`. Чтобы выйти из него, введи `deactivate`.


Для проверки, какие Python и `pip` используются в терминале, запусти:

```text
python --version
python -m pip --version
```

У команды `python -m pip` есть преимущество: она запускает `pip`, связанный именно с тем интерпретатором `python`, который будет запускать проект.


Обычно виртуальное окружение не добавляют в Git. Создай файл `.gitignore` в корне проекта и добавь туда:

```text
.venv/
__pycache__/
```

`__pycache__` содержит служебные файлы Python, которые он создаёт автоматически.


## 6. Устанавливаем и сохраняем зависимости

`pip` устанавливает внешние пакеты. Сначала активируй окружение, затем установи пакет командой:

```text
python -m pip install requests
```

`requests` здесь приведён как пример стороннего пакета. Для примеров в этом блокноте он не нужен, поэтому устанавливать его не обязательно.


Посмотреть сведения об установленном пакете можно командой:

```text
python -m pip show requests
```

Показать список установленных пакетов:

```text
python -m pip list
```


Чтобы записать текущие зависимости проекта в `requirements.txt`, выполни:

```text
python -m pip freeze > requirements.txt
```

Другой человек сможет установить эти версии в своём активном окружении:

```text
python -m pip install -r requirements.txt
```

В маленьком учебном проекте можно сначала написать `requirements.txt` вручную. Не добавляй в него библиотеки, которые проект не использует.


## 7. Небольшой проект из двух модулей

Создадим мини-проект викторины. В этом примере блокнот записывает файл-модуль с функцией, а затем импортирует её.


In [ ]:
%%writefile quiz_tools.py
def check_answer(answer, correct_answer):
    return answer.strip().lower() == correct_answer.strip().lower()


Теперь используем функцию из нового файла:


In [ ]:
from quiz_tools import check_answer

answer = input("Сколько будет 6 × 7? ")
if check_answer(answer, "42"):
    print("Верно!")
else:
    print("Попробуй ещё раз.")

В настоящем проекте эти файлы лежали бы рядом в папке проекта, а не создавались бы ячейками блокнота. Когда кода станет больше, вынеси разные части программы в отдельные модули и добавь короткий `README.md` с командой запуска.


## Домашние задания

### Базовый уровень

1. Создай модуль `text_tools.py` с функцией, которая возвращает строку в верхнем регистре. Импортируй её в отдельном файле.
2. Добавь в главную программу функцию `main()` и защиту `if __name__ == "__main__":`.
3. Создай виртуальное окружение для учебной папки и проверь версию Python и `pip`.

### Средний уровень

4. Раздели свою программу на два модуля: ввод/вывод и вычисления. Импортируй нужные функции из одного файла в другой.
5. Создай пакет с `__init__.py` и двумя модулями. Запусти один из них из корня проекта командой `python -m имя_пакета.модуль`.
6. Добавь внешний пакет в активное окружение, сохрани зависимости в `requirements.txt`, затем установи их в новом окружении.

### Задание со звёздочкой ★

Собери мини-проект викторины: отдельный модуль хранит вопросы, другой проверяет ответы, третий запускает игру. Добавь точку входа и краткий README с инструкциями по запуску.


## Короткое повторение

- Модуль — файл `.py`, который можно импортировать.
- Пакет — папка, объединяющая связанные модули.
- `if __name__ == "__main__":` защищает точку входа при импорте.
- `python -m package.module` запускает модуль из корня проекта как часть пакета.
- `venv` создаёт изолированное окружение для зависимостей проекта.
- Активируй окружение перед установкой пакетов.
- `python -m pip install ...` связывает `pip` с выбранным Python.
- `requirements.txt` помогает повторно установить зависимости.

### Проверь себя

1. Зачем делить программу на модули?
2. Что произойдёт с блоком `if __name__ == "__main__":` при импорте?
3. Почему активированное окружение помогает проектам не мешать друг другу?
4. Для чего нужен `requirements.txt`?
5. Из какой папки запускать `python -m quiz.main`?
